<a href="https://colab.research.google.com/github/HafizRizqi/MachineLearning_2026/blob/main/JS09/JS09_Tugas_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HAFIZ RIZQI HERNANDA - 244107020154 - TI_3C

#🎾Tugas Lab 1
1. Lakukan percobaan pada metric distance yang berbeda, 1000 vs 1.000.000 data, 2D vs 5D data, untuk algoritma,

ANNOY

FAISS

HNSW

Catat performansinya dalam bentuk tabel, misa

Langkah 1 – Instalasi library

In [1]:
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520038 sha256=5b5d16558cb1e4f1a601f7c7a27388043cd7ff51b7c4abd0e4281fd77363e777
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 30.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679795 sha256=ccb3639ad81c623650ed2743e554e7b039373ccac0c309fefa613350c69d2ced
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705

Langkah 2 – Import library

In [2]:
import numpy as np
import time
import matplotlib.pyplot as plt

from annoy import AnnoyIndex
import faiss
import hnswlib

Langkah 3 – Membuat fungsi normalisasi dan dataset

In [3]:
# Fungsi normalisasi data untuk Cosine Similarity
def normalize_data(data):
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / np.maximum(norms, 1e-10)


# Fungsi membuat dataset
def create_dataset(n_data, dim, metric, seed=42):

    np.random.seed(seed)

    # Membuat dataset random
    X = np.random.random((n_data, dim)).astype(np.float32)

    # Membuat 20 query
    query = np.random.random((20, dim)).astype(np.float32)

    # Normalisasi jika menggunakan Cosine
    if metric == "cosine":
        X = normalize_data(X)
        query = normalize_data(query)

    return X, query


print("Fungsi dataset berhasil dibuat.")

Fungsi dataset berhasil dibuat.


Langkah 4 – Membuat fungsi pengujian ANNOY

In [4]:
def run_annoy(X, query, metric, k=10):

    dim = X.shape[1]

    # Menentukan metric
    if metric == "euclidean":
        annoy_metric = "euclidean"
    else:
        annoy_metric = "angular"

    # Membuat index
    index = AnnoyIndex(dim, annoy_metric)

    # Menghitung waktu pembangunan index
    start = time.perf_counter()

    for i, vector in enumerate(X):
        index.add_item(i, vector)

    index.build(10)

    build_time = time.perf_counter() - start

    # Menghitung waktu pencarian
    start = time.perf_counter()

    results = [
        index.get_nns_by_vector(q, k)
        for q in query
    ]

    query_time = (time.perf_counter() - start) / len(query)

    return build_time, query_time, results

Langkah 5 – Membuat fungsi pengujian FAISS

In [5]:
def run_faiss(X, query, metric, k=10):

    dim = X.shape[1]
    n_data = len(X)

    # Menentukan jumlah cluster
    nlist = 10 if n_data == 1000 else 100

    # Membuat quantizer
    quantizer = faiss.IndexFlatL2(dim) if metric == "euclidean" \
        else faiss.IndexFlatIP(dim)

    # Menentukan metric FAISS
    if metric == "euclidean":
        index = faiss.IndexIVFFlat(
            quantizer, dim, nlist, faiss.METRIC_L2
        )
    else:
        index = faiss.IndexIVFFlat(
            quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT
        )

    # Membangun index
    start = time.perf_counter()

    index.train(X)
    index.add(X)

    build_time = time.perf_counter() - start

    # Menentukan jumlah cluster yang diperiksa
    index.nprobe = min(10, nlist)

    # Melakukan pencarian
    start = time.perf_counter()

    distances, indices = index.search(query, k)

    query_time = (time.perf_counter() - start) / len(query)

    return build_time, query_time, indices

Langkah 6 – Membuat fungsi pengujian HNSW

In [6]:
def run_hnsw(X, query, metric, k=10):

    dim = X.shape[1]
    n_data = len(X)

    # Menentukan metric
    if metric == "euclidean":
        space = "l2"
    else:
        space = "cosine"

    # Membuat index
    index = hnswlib.Index(
        space=space,
        dim=dim
    )

    # Membangun index
    start = time.perf_counter()

    index.init_index(
        max_elements=n_data,
        ef_construction=200,
        M=16
    )

    index.add_items(X)

    build_time = time.perf_counter() - start

    # Parameter pencarian
    index.set_ef(50)

    # Melakukan pencarian
    start = time.perf_counter()

    labels, distances = index.knn_query(
        query,
        k=k
    )

    query_time = (time.perf_counter() - start) / len(query)

    return build_time, query_time, labels

Langkah 7 – Menentukan skenario pengujian

In [7]:
scenarios = [
    (1000, 2, "euclidean"),
    (1000, 2, "cosine"),
    (1000000, 5, "euclidean"),
    (1000000, 5, "cosine")
]

print("Jumlah skenario:", len(scenarios))

Jumlah skenario: 4


Langkah 8 – Menjalankan ANNOY, FAISS, dan HNSW

In [8]:
results = []

for n_data, dim, metric in scenarios:

    print("\n================================")
    print(f"DATA: {n_data:,}")
    print(f"DIMENSI: {dim}D")
    print(f"METRIC: {metric.upper()}")
    print("================================")

    # Membuat dataset
    X, query = create_dataset(
        n_data, dim, metric
    )

    # ANNOY
    annoy_build, annoy_query, annoy_result = run_annoy(
        X, query, metric
    )

    # FAISS
    faiss_build, faiss_query, faiss_result = run_faiss(
        X, query, metric
    )

    # HNSW
    hnsw_build, hnsw_query, hnsw_result = run_hnsw(
        X, query, metric
    )

    # Menyimpan hasil
    results.append({
        "Data": n_data,
        "Dimensi": dim,
        "Metric": metric,
        "ANNOY Build": annoy_build,
        "ANNOY Query": annoy_query,
        "FAISS Build": faiss_build,
        "FAISS Query": faiss_query,
        "HNSW Build": hnsw_build,
        "HNSW Query": hnsw_query
    })

    print("ANNOY :", annoy_build, "detik |",
          annoy_query * 1000, "ms/query")

    print("FAISS :", faiss_build, "detik |",
          faiss_query * 1000, "ms/query")

    print("HNSW  :", hnsw_build, "detik |",
          hnsw_query * 1000, "ms/query")


DATA: 1,000
DIMENSI: 2D
METRIC: EUCLIDEAN
ANNOY : 0.01968319899992821 detik | 0.026739050002788645 ms/query
FAISS : 0.001218300999994426 detik | 0.01205890000051113 ms/query
HNSW  : 0.04966367400004401 detik | 0.018014900001617207 ms/query

DATA: 1,000
DIMENSI: 2D
METRIC: COSINE
ANNOY : 0.03022808299999724 detik | 0.027594650003948118 ms/query
FAISS : 0.00110462500003905 detik | 0.013350649999210873 ms/query
HNSW  : 0.04395911700009947 detik | 0.016569650000519687 ms/query

DATA: 1,000,000
DIMENSI: 5D
METRIC: EUCLIDEAN
ANNOY : 20.493634021999924 detik | 0.063819849998481 ms/query
FAISS : 0.20107614199991986 detik | 0.6732444000022042 ms/query
HNSW  : 178.02414760399995 detik | 0.06900605000055293 ms/query

DATA: 1,000,000
DIMENSI: 5D
METRIC: COSINE
ANNOY : 29.177993544999936 detik | 0.07222214999842436 ms/query
FAISS : 0.3124865890000592 detik | 0.9994038000002091 ms/query
HNSW  : 169.11421733499992 detik | 0.04853589999811447 ms/query


Langkah 9 – Membuat tabel perbandingan

In [9]:
import pandas as pd

df = pd.DataFrame(results)

# Membuat tabel ringkas
table = pd.DataFrame({
    "Jumlah Data/Dimensi": [
        f"{r['Data']:,}/{r['Dimensi']}D - {r['Metric']}"
        for r in results
    ],

    "ANNOY": [
        f"{r['ANNOY Query']*1000:.4f} ms"
        for r in results
    ],

    "FAISS": [
        f"{r['FAISS Query']*1000:.4f} ms"
        for r in results
    ],

    "HNSW": [
        f"{r['HNSW Query']*1000:.4f} ms"
        for r in results
    ]
})

display(table)

,Jumlah Data/Dimensi,ANNOY,FAISS,HNSW
0,"1,000/2D - euclidean",0.0267 ms,0.0121 ms,0.0180 ms
1,"1,000/2D - cosine",0.0276 ms,0.0134 ms,0.0166 ms
2,"1,000,000/5D - euclidean",0.0638 ms,0.6732 ms,0.0690 ms
3,"1,000,000/5D - cosine",0.0722 ms,0.9994 ms,0.0485 ms


Langkah 10 – Membandingkan waktu pembangunan index

In [10]:
build_table = pd.DataFrame({
    "Jumlah Data/Dimensi": [
        f"{r['Data']:,}/{r['Dimensi']}D - {r['Metric']}"
        for r in results
    ],
    "ANNOY (detik)": [
        round(r["ANNOY Build"], 4) for r in results
    ],
    "FAISS (detik)": [
        round(r["FAISS Build"], 4) for r in results
    ],
    "HNSW (detik)": [
        round(r["HNSW Build"], 4) for r in results
    ]
})

display(build_table)

,Jumlah Data/Dimensi,ANNOY (detik),FAISS (detik),HNSW (detik)
0,"1,000/2D - euclidean",0.0197,0.0012,0.0497
1,"1,000/2D - cosine",0.0302,0.0011,0.0440
2,"1,000,000/5D - euclidean",20.4936,0.2011,178.0241
3,"1,000,000/5D - cosine",29.1780,0.3125,169.1142


# Analisis Hasil
**Percobaan 1 – 1.000 Data, 2D, Euclidean Distance**

Pada percobaan pertama, dilakukan pengujian terhadap 1.000 data dengan dimensi 2D menggunakan Euclidean Distance. Berdasarkan hasil pengujian, FAISS memperoleh waktu pencarian paling cepat, yaitu sebesar 0,0121 ms/query. Sementara itu, HNSW membutuhkan waktu 0,0180 ms/query dan ANNOY membutuhkan waktu 0,0267 ms/query.

Pada proses pembangunan index, FAISS juga menunjukkan waktu paling cepat, yaitu 0,0012 detik. ANNOY membutuhkan waktu 0,0197 detik, sedangkan HNSW membutuhkan waktu 0,0497 detik.

Dari hasil tersebut, dapat diketahui bahwa FAISS memiliki performa terbaik pada dataset berukuran kecil dengan metric Euclidean, baik dari segi pembangunan index maupun waktu pencarian.

**Percobaan 2 – 1.000 Data, 2D, Cosine Similarity**

Pada percobaan kedua, digunakan jumlah data sebanyak 1.000 dengan dimensi 2D, tetapi metric distance diganti menjadi Cosine Similarity.

Hasil pengujian menunjukkan bahwa FAISS kembali memperoleh waktu pencarian paling cepat, yaitu 0,0134 ms/query. HNSW memperoleh waktu 0,0166 ms/query, sedangkan ANNOY memperoleh waktu 0,0276 ms/query.

Untuk pembangunan index, FAISS membutuhkan waktu 0,0011 detik, ANNOY membutuhkan waktu 0,0302 detik, dan HNSW membutuhkan waktu 0,0440 detik.

Jika dibandingkan dengan percobaan pertama, penggunaan Cosine Similarity menyebabkan waktu pencarian ketiga algoritma sedikit meningkat. Hal ini menunjukkan bahwa perubahan metric dapat memengaruhi performa pencarian, meskipun jumlah data dan dimensinya tetap sama.

**Percobaan 3 – 1.000.000 Data, 5D, Euclidean Distance**

Pada percobaan ketiga, jumlah data ditingkatkan menjadi 1.000.000 data dengan dimensi 5D menggunakan Euclidean Distance. Percobaan ini bertujuan mengetahui kemampuan ketiga algoritma dalam menangani dataset berukuran besar.

Berdasarkan hasil pengujian, ANNOY memperoleh waktu pencarian paling cepat, yaitu 0,0638 ms/query. HNSW memperoleh waktu 0,0690 ms/query, sedangkan FAISS membutuhkan waktu 0,6732 ms/query.

Namun, pada proses pembangunan index, FAISS menunjukkan hasil paling cepat dengan waktu 0,2011 detik. ANNOY membutuhkan waktu 20,4936 detik, sedangkan HNSW membutuhkan waktu 178,0241 detik.

Dari hasil tersebut, dapat diketahui bahwa FAISS memiliki keunggulan dalam pembangunan index, sedangkan ANNOY memiliki waktu pencarian paling cepat pada percobaan ini. HNSW membutuhkan waktu pembangunan index paling lama dibandingkan kedua algoritma lainnya.

**Percobaan 4 – 1.000.000 Data, 5D, Cosine Similarity**

Pada percobaan keempat, digunakan 1.000.000 data dengan dimensi 5D menggunakan Cosine Similarity.

Hasil pengujian menunjukkan bahwa HNSW memperoleh waktu pencarian paling cepat, yaitu 0,0485 ms/query. ANNOY membutuhkan waktu 0,0722 ms/query, sedangkan FAISS membutuhkan waktu 0,9994 ms/query.

Pada pembangunan index, FAISS kembali memperoleh waktu paling cepat, yaitu 0,3125 detik. ANNOY membutuhkan waktu 29,1780 detik, sedangkan HNSW membutuhkan waktu 169,1142 detik.

Hasil ini menunjukkan bahwa HNSW memiliki performa pencarian terbaik pada dataset berukuran besar dengan metric Cosine Similarity. Sementara itu, FAISS memiliki keunggulan dalam hal kecepatan pembangunan index.

# Kesimpulan

Berdasarkan pembelajaran hari ini mengenai Approximate Nearest Neighbors (ANN), telah dilakukan pengujian terhadap tiga algoritma, yaitu ANNOY, FAISS, dan HNSW dengan menggunakan dua jenis metric distance, yaitu Euclidean Distance dan Cosine Similarity.

Pengujian dilakukan pada empat skenario, yaitu 1.000 data dengan dimensi 2D menggunakan Euclidean Distance, 1.000 data dengan dimensi 2D menggunakan Cosine Similarity, 1.000.000 data dengan dimensi 5D menggunakan Euclidean Distance, serta 1.000.000 data dengan dimensi 5D menggunakan Cosine Similarity.

Berdasarkan hasil percobaan, diperoleh beberapa kesimpulan:

1. Pada dataset berukuran kecil (1.000 data, 2D), FAISS memiliki performa paling baik karena menghasilkan waktu pembangunan index dan waktu pencarian paling cepat dibandingkan ANNOY dan HNSW.

2. Pada dataset berukuran besar (1.000.000 data, 5D) dengan Euclidean Distance, ANNOY menghasilkan waktu pencarian paling cepat, yaitu 0,0638 ms/query. Sementara itu, FAISS memiliki waktu pembangunan index paling cepat, yaitu 0,2011 detik.

3. Pada dataset berukuran besar dengan Cosine Similarity, HNSW menghasilkan waktu pencarian paling cepat, yaitu 0,0485 ms/query. Namun, FAISS tetap unggul dalam proses pembangunan index dengan waktu 0,3125 detik.

4. Penggunaan metric distance yang berbeda memengaruhi performa setiap algoritma. Hal ini terlihat dari perubahan waktu pencarian dan pembangunan index pada masing-masing percobaan.

5. Peningkatan jumlah data dan dimensi memengaruhi waktu pembangunan index. Hal ini terlihat terutama pada ANNOY dan HNSW ketika jumlah data meningkat dari 1.000 menjadi 1.000.000 data.

Secara keseluruhan, tidak ada satu algoritma yang selalu memiliki performa terbaik pada seluruh skenario pengujian. FAISS unggul dalam kecepatan pembangunan index, ANNOY menunjukkan performa pencarian yang baik pada penggunaan Euclidean Distance dengan dataset besar, sedangkan HNSW menghasilkan waktu pencarian terbaik pada dataset besar dengan Cosine Similarity.